# Thử nghiệm CatBoost: dự báo điểm Toán mô phỏng

Notebook giới thiệu hồi quy bằng `CatBoostRegressor`. Tệp 10.000 hồ sơ gốc không có nhãn điểm thi, vì vậy ví dụ dùng `dulieuhocsinh_50000_mo_phong.csv` để dự báo `diem_thi_toan_mo_phong`.

> Điểm đích là nhãn mô phỏng, không phải điểm thi thật. Kết quả chỉ phản ánh khả năng học quy luật trong bộ dữ liệu này.

Nếu thiếu thư viện: `pip install catboost pandas scikit-learn`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_SEED = 42
TARGET = "diem_thi_toan_mo_phong"
DATA_FILE = "dulieuhocsinh_50000_mo_phong.csv"
possible_paths = [Path.cwd() / DATA_FILE, Path.cwd().parent / DATA_FILE]
DATA_PATH = next((p for p in possible_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(f"Không tìm thấy {DATA_FILE}; mở notebook từ thư mục dự án hoặc analys/.")
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df.columns = df.columns.str.strip()
print(f"Tệp: {DATA_PATH.resolve()}")
print(f"Kích thước: {df.shape[0]:,} dòng × {df.shape[1]} cột")
print("Số dòng theo phân vùng:")
print(df["tap_du_lieu"].value_counts().sort_index().to_string())
print(f"Nhãn {TARGET}: min={df[TARGET].min():.2f}, max={df[TARGET].max():.2f}, trung bình={df[TARGET].mean():.2f}")

Tệp: D:\Nam4\MLAI\dulieuhocsinh_50000_mo_phong.csv
Kích thước: 50,000 dòng × 131 cột
Số dòng theo phân vùng:
tap_du_lieu
test          10000
train         30000
validation    10000
Nhãn diem_thi_toan_mo_phong: min=0.00, max=10.00, trung bình=4.48


## CatBoost là gì?

CatBoost là thuật toán boosting trên cây quyết định. Mỗi cây mới tập trung sửa lỗi còn lại của tổ hợp cây trước; dự đoán cuối là tổng đóng góp của các cây. `CatBoostRegressor` với `loss_function="RMSE"` học để giảm sai số bình phương.

CatBoost cũng hỗ trợ đặc trưng phân loại và có cơ chế giảm thiên lệch trong boosting. Ví dụ này dùng đặc trưng số, chưa minh họa cột phân loại.

Ta dùng lịch sử điểm, điểm thi thử Toán, giờ tự học và chuyên cần. Mốc đơn giản để so sánh là lấy `toan_lop9` làm dự đoán điểm thi.

In [2]:
group_sets = {name: set(part["nhom_ho_so_goc"].unique()) for name, part in df.groupby("tap_du_lieu")}
for left, right in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    overlap = group_sets[left] & group_sets[right]
    print(f"Nhóm trùng giữa {left} và {right}: {len(overlap)}")
    assert not overlap, f"Rò rỉ nhóm hồ sơ giữa {left} và {right}"
train_df = df.loc[df["tap_du_lieu"] == "train"].copy()
valid_df = df.loc[df["tap_du_lieu"] == "validation"].copy()
test_df = df.loc[df["tap_du_lieu"] == "test"].copy()
print(f"Số nhóm: train={train_df['nhom_ho_so_goc'].nunique():,}, validation={valid_df['nhom_ho_so_goc'].nunique():,}, test={test_df['nhom_ho_so_goc'].nunique():,}")

Nhóm trùng giữa train và validation: 0
Nhóm trùng giữa train và test: 0
Nhóm trùng giữa validation và test: 0
Số nhóm: train=6,000, validation=2,000, test=2,000


## Nạp và kiểm tra dữ liệu

Dùng sẵn các tập train/validation/test trong CSV. Kiểm tra để bảo đảm một `nhom_ho_so_goc` không xuất hiện ở nhiều tập, tránh biến thể của cùng hồ sơ lọt sang cả train lẫn test.

In [3]:
academic_cols = [c for c in df.columns if c.endswith(tuple(f"_lop{g}" for g in range(6, 10)))]
feature_cols = academic_cols + ["gio_tu_hoc_tuan", "ti_le_chuyen_can", "diem_thi_thu_toan"]
X_train, y_train = train_df[feature_cols], train_df[TARGET]
X_valid, y_valid = valid_df[feature_cols], valid_df[TARGET]
X_test, y_test = test_df[feature_cols], test_df[TARGET]
print(f"Số đặc trưng: {len(feature_cols)}")
print(f"Số dòng: train={len(train_df):,}, validation={len(valid_df):,}, test={len(test_df):,}")
model = CatBoostRegressor(iterations=1000, depth=8, learning_rate=0.05,
    loss_function="RMSE", eval_metric="RMSE", random_seed=RANDOM_SEED,
    verbose=False, allow_writing_files=False)
model.fit(X_train, y_train, eval_set=(X_valid, y_valid),
    early_stopping_rounds=100, verbose=False)
print(f"Số cây đã dùng: {model.tree_count_}")
print(f"Vòng tốt nhất (đếm từ 0): {model.get_best_iteration()}")

Số đặc trưng: 55
Số dòng: train=30,000, validation=10,000, test=10,000
Số cây đã dùng: 256
Vòng tốt nhất (đếm từ 0): 255


## Chọn đặc trưng và huấn luyện

Mô hình chọn số cây bằng validation với early stopping. Test chỉ dùng để đánh giá cuối. Loại mã học sinh, nhóm hồ sơ, thông tin nguyện vọng, điểm chuẩn, tổng điểm và kết quả tuyển sinh khỏi đầu vào.

In [4]:
def calculate_metrics(actual, predicted):
    return {"MAE": mean_absolute_error(actual, predicted),
            "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
            "R²": r2_score(actual, predicted)}
rows = []
for split_name, part, X, y in [("validation", valid_df, X_valid, y_valid), ("test", test_df, X_test, y_test)]:
    rows.append({"Tập": split_name, "Mô hình": "CatBoost", **calculate_metrics(y, model.predict(X))})
    rows.append({"Tập": split_name, "Mô hình": "Mốc: toan_lop9", **calculate_metrics(y, part["toan_lop9"])})
results = pd.DataFrame(rows)
print(results.to_string(index=False, formatters={"MAE": "{:.3f}".format, "RMSE": "{:.3f}".format, "R²": "{:.3f}".format}))

       Tập        Mô hình   MAE  RMSE    R²
validation       CatBoost 0.733 0.926 0.875
validation Mốc: toan_lop9 1.021 1.303 0.753
      test       CatBoost 0.755 0.951 0.866
      test Mốc: toan_lop9 1.065 1.352 0.730


## Đánh giá

- **MAE**: độ lệch tuyệt đối trung bình, tính theo điểm.
- **RMSE**: phạt lỗi lớn mạnh hơn MAE.
- **R²**: mức biến thiên được giải thích so với dự đoán bằng trung bình.

So sánh với `toan_lop9` trên cùng tập để xem mô hình có cải thiện dự báo trong bộ mô phỏng hay không.

In [5]:
importance = pd.Series(model.feature_importances_, index=feature_cols, name="Mức quan trọng").sort_values(ascending=False)
print(importance.head(10).to_string(float_format=lambda x: f"{x:.2f}%"))

toan_lop9                58.60%
toan_lop8                18.75%
diem_thi_thu_toan         7.72%
gio_tu_hoc_tuan           1.54%
ti_le_chuyen_can          0.51%
toan_lop7                 0.44%
toan_lop6                 0.44%
giao_duc_the_chat_lop8    0.43%
lich_su_dia_ly_lop8       0.41%
tin_hoc_lop7              0.36%


## Tầm quan trọng của đặc trưng

Mức quan trọng là cách mô hình phân bổ đóng góp dự báo giữa các cột. Đây không phải quan hệ nhân quả và không chứng minh can thiệp vào đặc trưng sẽ làm điểm tăng.

In [6]:
preview = pd.DataFrame({
    "Điểm Toán lớp 9 (mốc)": test_df["toan_lop9"].head(5).to_numpy(),
    "Điểm thi mô phỏng": y_test.head(5).to_numpy(),
    "Dự đoán CatBoost": model.predict(X_test.head(5)),
})
print(preview.round(2).to_string(index=False))

 Điểm Toán lớp 9 (mốc)  Điểm thi mô phỏng  Dự đoán CatBoost
                   2.0               1.04              1.81
                   8.0               5.60              5.40
                   4.0               1.27              2.51
                   3.0               6.28              4.26
                   5.0               4.86              5.27


## Diễn giải kết quả chạy

CatBoost dừng tại **256 cây** (vòng tốt nhất 255, đếm từ 0). Trên 10.000 dòng test, kết quả là **MAE 0,755**, **RMSE 0,951**, **R² 0,866**. Mốc `toan_lop9` lần lượt có **MAE 1,065** và **RMSE 1,352**. Trong lần chạy này, CatBoost giảm MAE khoảng **29%** và RMSE khoảng **30%**.

`toan_lop9` quan trọng nhất (58,6%), kế đến `toan_lop8` (18,7%) và `diem_thi_thu_toan` (7,7%). Điều này cho thấy các điểm gần thời điểm thi mang tín hiệu dự báo lớn trong dữ liệu mô phỏng.

**Giới hạn:** tài liệu mô tả nhãn thi mô phỏng được tạo chủ yếu từ điểm lớp 8–9, giờ tự học, chuyên cần và nhiễu. Kết quả tốt có thể phản ánh mô hình học lại quy tắc sinh nhãn; chưa thể suy ra độ chính xác với học sinh thật hay hiệu quả tư vấn tuyển sinh.

## Thử nghiệm tiếp theo

Có thể lần lượt đổi nhãn sang `diem_thi_ngu_van_mo_phong` và `diem_thi_ngoai_ngu_mo_phong`, rồi so sánh CatBoost với SVR trên cùng đặc trưng, phân vùng, chỉ số và quy trình chọn tham số.